In [3]:
import nltk
nltk.download('stopwords')

from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


from transformers import pipeline

import pandas as pd



[nltk_data] Downloading package stopwords to C:\Users\VIVOBBOK
[nltk_data]     16\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 1. Create a Sample Dataset
Since there is no external dataset provided, we will create a small sample dataset with text and corresponding emotion labels.

In [4]:
# Create a sample dataset for emotion detection
data = {
    'text': [
        'I am so happy and joyful today!',
        'This is the worst day of my life, I am so sad.',
        'I am furious and angry at this situation!',
        'I feel terrified and scared by the loud noise.',
        'What a wonderful surprise, I am amazed!',
        'I feel so depressed and lonely.',
        'This makes me very mad!',
        'I am absolutely delighted!'
    ],
    'emotion': [
        'joy', 'sadness', 'anger', 'fear', 'surprise', 'sadness', 'anger', 'joy'
    ]
}
df = pd.DataFrame(data)
display(df.head())

,text,emotion
0,I am so happy and joyful today!,joy
1,"This is the worst day of my life, I am so sad.",sadness
2,I am furious and angry at this situation!,anger
3,I feel terrified and scared by the loud noise.,fear
4,"What a wonderful surprise, I am amazed!",surprise


## 2. Text Preprocessing
We will clean the text by lowercasing it, removing punctuation, and filtering out stopwords using NLTK.

In [5]:
# Preprocessing function using NLTK
import re
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Remove stopwords
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

df['clean_text'] = df['text'].apply(preprocess_text)
display(df[['text', 'clean_text']].head())

,text,clean_text
0,I am so happy and joyful today!,happy joyful today
1,"This is the worst day of my life, I am so sad.",worst day life sad
2,I am furious and angry at this situation!,furious angry situation
3,I feel terrified and scared by the loud noise.,feel terrified scared loud noise
4,"What a wonderful surprise, I am amazed!",wonderful surprise amazed


## 3. Feature Extraction and Train-Test Split
Next, we use `TfidfVectorizer` to convert text into numerical features, and split the data into training and test sets.

In [6]:
# Splitting data and feature extraction using TF-IDF
X_train, X_test, y_train, y_test = train_test_split(df['clean_text'], df['emotion'], test_size=0.25, random_state=42)

vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training data shape:", X_train_tfidf.shape)
print("Testing data shape:", X_test_tfidf.shape)

Training data shape: (6, 18)
Testing data shape: (2, 18)


## 4. Train Multinomial Naive Bayes Model
We train a baseline model using `MultinomialNB`.

In [7]:
# Train Multinomial Naive Bayes model
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

# Predictions and Evaluation
y_pred = nb_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"Naive Bayes Model Accuracy: {accuracy * 100:.2f}%")

Naive Bayes Model Accuracy: 0.00%


## 5. Emotion Detection using HuggingFace Transformers (Zero-Shot / Pipeline)
For advanced emotion detection, we can use a pre-trained model from HuggingFace.

In [8]:
# Emotion detection using HuggingFace Transformers Pipeline
# We use a pre-trained model fine-tuned for emotion detection
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base")

sample_texts = [
    "I am so happy that I passed the exam!",
    "I feel so sad and lonely today.",
    "I am furious about the bad service!"
]

predictions = classifier(sample_texts)
for text, pred in zip(sample_texts, predictions):
    print(f"Text: {text}")
    print(f"Emotion: {pred['label']}, Score: {pred['score']:.4f}\n")

config.json: 0.00B [00:00, ?B/s]

c:\Users\VIVOBBOK 16\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VIVOBBOK 16\.cache\huggingface\hub\models--j-hartmann--emotion-english-distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' p

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/j-hartmann/emotion-english-distilroberta-base/resolve/main/pytorch_model.bin: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...


pytorch_model.bin:  51%|#####1    | 168M/329M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cpu


Text: I am so happy that I passed the exam!
Emotion: joy, Score: 0.9879

Text: I feel so sad and lonely today.
Emotion: sadness, Score: 0.9875

Text: I am furious about the bad service!
Emotion: anger, Score: 0.9855

